# AIMO3 Math Solver - Notebook V2

This notebook contains all the code needed to run the AIMO3 math solver without relying on the model library.
It includes:
- Utility functions for text extraction and parsing
- Prompt templates
- Stream collection for both API and local models
- Tool implementations (sandbox)
- The KaggleSolver class
- Configuration and prediction service setup

In [7]:
# =============================================================================
# Section 1: Imports and Dependencies
# =============================================================================
import json
import re
import os
import time
import queue
import contextlib
import threading
from abc import ABC, abstractmethod
from typing import Any, Dict, Optional, List
from collections import Counter
from threading import Thread

import yaml
import pandas as pd
import polars as pl

from openai import OpenAI
from openai.types.chat import ChatCompletionMessage

from transformers import AutoModelForCausalLM, AutoTokenizer, TextIteratorStreamer

# import aimo3_math.kaggle_evaluation.aimo_3_inference_server

In [8]:
# =============================================================================
# Section 2: Format Utilities - Text Extraction and Parsing
# =============================================================================
# These functions are used to extract thinking content, tool calls, and answers from model output

def extract_think(text: str) -> Optional[str]:
    """Extract thinking content between \u3008think\u3009 and \u3009/think\u3009 tokens."""
    pattern = r'\u3008think\u3009(.*?)\u3009/think\u3009'
    matches = re.findall(pattern, text, re.DOTALL)
    if len(matches) == 0:
        return None
    return matches[0].strip()

def exclude_think(text: str) -> str:
    """Remove thinking content from text."""
    if "\u3009/think\u3009" not in text:
        return text
    splits = text.split("\u3009/think\u3009")
    return splits[-1]

def exclude_tool_call(text: str) -> str:
    """Remove tool call XML tags from text."""
    if "<tool_call>" not in text:
        return text
    splits = text.split("<tool_call>")
    return splits[0]

def extract_tool_call(text: str) -> Optional[str]:
    """Extract tool call content from <tool_call> XML tags."""
    pattern = r'<tool_call>(.*?)</tool_call>'
    matches = re.findall(pattern, text, re.DOTALL)
    if len(matches) == 0:
        return None
    return matches[-1].strip()

def extract_answer(text: str) -> int:
    """Extract final answer from \\boxed{} LaTeX format."""
    text = exclude_think(text)
    pattern = r"\\boxed{([^{}]+)}"
    matches = re.findall(pattern, text)
    if matches:
        try:
            ans = matches[-1].strip()
            return int(ans)
        except Exception:
            pass
    return 0

In [9]:
# =============================================================================
# Section 3: Prompt Templates
# =============================================================================
# System prompts and preference prompts for the math solver

SYSTEM_PROMPT = (
    'You are an elite mathematical problem solver with expertise at the International '
    'Mathematical Olympiad (IMO) level. Your goal is to find the correct answer through '
    'rigorous mathematical reasoning.\n\n'

    '# Problem-Solving Approach:\n'
    '1. UNDERSTAND: Carefully read and rephrase the problem in your own words. '
    'Identify what is given, what needs to be found, and any constraints.\n'
    '2. EXPLORE: Consider multiple solution strategies. Think about relevant theorems, '
    'techniques, patterns, or analogous problems. Don\'t commit to one approach immediately.\n'
    '3. PLAN: Select the most promising approach and outline key steps before executing.\n'
    '4. EXECUTE: Work through your solution methodically. Show all reasoning steps clearly.\n'
    '5. VERIFY: Check your answer by substituting back, testing edge cases, or using '
    'alternative methods. Ensure logical consistency throughout.\n\n'

    '# Mathematical Reasoning Principles:\n'
    '- Break complex problems into smaller, manageable sub-problems\n'
    '- Look for patterns, symmetries, and special cases that provide insight\n'
    '- Use concrete examples to build intuition before generalizing\n'
    '- Consider extreme cases and boundary conditions\n'
    '- If stuck, try working backwards from the desired result\n'
    '- Be willing to restart with a different approach if needed\n\n'

    '# Verification Requirements:\n'
    '- Cross-check arithmetic and algebraic manipulations\n'
    '- Verify that your solution satisfies all problem constraints\n'
    '- Test your answer with simple cases or special values when possible\n'
    '- Ensure dimensional consistency and reasonableness of the result\n\n'

    '# Output Format:\n'
    'The final answer must be a non-negative integer between 0 and 99999.\n'
    'Place your final numerical answer inside \\boxed{}, e.g., \\boxed{42}\n\n'

    'Think step-by-step and show your complete reasoning process. Quality of reasoning '
    'is as important as the final answer.'
)

PREFERENCE_PROMPT = (
    'You have access to `math`, `numpy`, and `sympy` for:\n\n'

    '# Symbolic Computation (sympy):\n'
    '- Algebraic manipulation and simplification\n'
    '- Solving equations and systems of equations\n'
    '- Symbolic differentiation and integration\n'
    '- Number theory functions (primes, divisors, modular arithmetic)\n'
    '- Polynomial operations and factorization\n'
    '- Working with mathematical expressions symbolically\n\n'

    '# Numerical Computation (numpy):\n'
    '- Array operations and linear algebra\n'
    '- Efficient numerical calculations for large datasets\n'
    '- Matrix operations and eigenvalue problems\n'
    '- Statistical computations\n\n'

    '# Mathematical Functions (math):\n'
    '- Standard mathematical functions (trig, log, exp)\n'
    '- Constants like pi and e\n'
    '- Basic operations for single values\n\n'

    'Best Practices:\n'
    '- Use sympy for exact symbolic answers when possible\n'
    '- Use numpy for numerical verification and large-scale computation\n'
    '- Combine symbolic and numerical approaches: derive symbolically, verify numerically\n'
    '- Document your computational strategy clearly\n'
    '- Validate computational results against known cases or theoretical bounds'
)

In [10]:
# =============================================================================
# Section 4: Stream Collection Functions
# =============================================================================
# Functions to collect and process streaming output from models

def collect_api_stream(stream, verbose=True):
    """Collect streaming output from API-based models (OpenAI-compatible)."""
    content = []
    reasoning_content = []
    tool_calls = {}
    is_answering = False
    print("\n" + "=" * 20 + "思考过程" + "=" * 20 + "\n")
    for chunk in stream:
        delta = chunk.choices[0].delta
        if hasattr(delta, "content") and delta.content:
            if verbose:
                if not is_answering:
                    print("\n" + "=" * 20 + "完整回复" + "=" * 20 + "\n")
                    is_answering = True
                print(delta.content, end="", flush=True)
            content.append(delta.content)
        if hasattr(delta, "reasoning_content") and delta.reasoning_content:
            if not is_answering and verbose:
                print(delta.reasoning_content, end="", flush=True)
            reasoning_content.append(delta.reasoning_content)
        if hasattr(delta, "tool_calls") and delta.tool_calls:
            for tool_call_chunk in delta.tool_calls:
                call_index = tool_call_chunk.index
                tool_call_chunk.function.arguments = tool_call_chunk.function.arguments or ""
                if call_index not in tool_calls:
                    tool_calls[call_index] = tool_call_chunk
                else:
                    tool_calls[call_index].function.arguments += tool_call_chunk.function.arguments
    content = "".join(content).strip()
    if len(content) == 0:
        content = None
    reasoning_content = "".join(reasoning_content).strip()
    if len(reasoning_content) == 0:
        reasoning_content = None
    if len(tool_calls) == 0:
        tool_calls = None
    else:
        tool_calls = [tool_call.to_dict() for tool_call in tool_calls.values()]
    return content, reasoning_content, tool_calls

def collect_model_stream(stream, verbose=True):
    """Collect streaming output from local models (transformers-based)."""
    reasoning_parts = []
    accumulated_text = ""
    
    print("\n" + "=" * 20 + "思考过程" + "=" * 20 + "\n")
    
    for chunk in stream:
        accumulated_text += chunk
    
    accumulated_text = accumulated_text.strip()
    
    # Extract reasoning content
    if "<think>" in accumulated_text and "</think>" in accumulated_text:
        start = accumulated_text.find("<think>") + len("<think>")
        end = accumulated_text.find("</think>")
        reasoning_content = accumulated_text[start:end]
        if verbose:
            print(reasoning_content, end="", flush=True)
        reasoning_parts.append(reasoning_content)
    
    if verbose:
        print("\n" + "=" * 20 + "完整回复" + "=" * 20 + "\n")
    
    content = exclude_tool_call(accumulated_text)
    content = exclude_think(content)
    reasoning_content = extract_think(accumulated_text)
    
    # Extract tool calls
    tool_call_str = extract_tool_call(accumulated_text)
    if tool_call_str is not None and len(tool_call_str) > 0:
        try:
            tool_calls = [json.loads(tool_call_str)]
        except json.JSONDecodeError:
            tool_calls = None
    else:
        tool_calls = None
    
    if content is None or len(content) == 0:
        content = None
    if reasoning_content is None or len(reasoning_content) == 0:
        reasoning_content = None
        
    if verbose and content:
        print(content, end="", flush=True)
    
    return content, reasoning_content, tool_calls

def construct_completion_message(content, reasoning_content, tool_calls):
    """Construct a ChatCompletionMessage object from collected stream data."""
    message = ChatCompletionMessage(
        role="assistant",
        content=content,
        reasoning_content=reasoning_content,
        tool_calls=tool_calls
    )
    return message

In [11]:
# =============================================================================
# Section 5: Tool Implementations
# =============================================================================
# Base tool class and sandbox implementation for code execution

class ToolBase(ABC):
    """Abstract base class for tools."""
    name = "default"

    @abstractmethod
    def __call__(self, *args, **kwargs): ...

    def get_tool_schema(self): ...


TOOL_PROMPT = (
    'Use this tool to execute Python code for:\n'
    '- Complex calculations that would be error-prone by hand\n'
    '- Numerical verification of analytical results\n'
    '- Generating examples or testing conjectures\n'
    '- Visualizing problem structure when helpful\n'
    '- Brute-force verification for small cases\n\n'

    'The environment is a stateful Jupyter notebook. Code persists between executions.\n'
    'Always use print() to display results. Write clear, well-commented code.\n\n'

    'Remember: Code should support your mathematical reasoning, not replace it. '
    'Explain what you\'re computing and why before running code.'
)


class AIMO3Sandbox(ToolBase):
    """
    AIMO3 Sandbox environment for safely executing Python code.
    Supports context manager (with statement) for automatic resource cleanup.
    """

    _port_lock = threading.Lock()
    _next_port = 50000
    name = "python_sandbox"

    @classmethod
    def _get_next_ports(cls, count: int = 5) -> List[int]:
        """Dynamically allocate available ports to avoid multi-instance conflicts."""
        with cls._port_lock:
            ports = list(range(cls._next_port, cls._next_port + count))
            cls._next_port += count
            return ports

    def __init__(self, timeout: float):
        self._default_timeout = timeout
        self._km = None
        self._client = None
        self._owns_kernel = False
        
        # Initialize kernel
        self._setup_kernel()
        # Preload base libraries
        self.reset()

    def get_tool_schema(self):
        """Get the tool schema for LLM function calling."""
        schema = {
            "type": "function",
            "function": {
                "name": "python_sandbox",
                "description": TOOL_PROMPT,
                "parameters": {
                    "type": "object",
                    "properties": {
                        "code": {
                            "type": "string",
                            "description": "Python script to execute",
                        },
                        "timeout": {
                            "type": "float",
                            "description": "running timeout in seconds",
                        },
                    },
                    "required": ["code"],
                },
                "strict": True
            }
        }
        return schema

    def _setup_kernel(self):
        """Initialize Jupyter kernel configuration."""
        from jupyter_client import KernelManager
        
        ports = self._get_next_ports(5)
        env = os.environ.copy()
        env.update({
            'PYDEVD_DISABLE_FILE_VALIDATION': '1',
            'PYDEVD_WARN_EVALUATION_TIMEOUT': '0',
            'JUPYTER_PLATFORM_DIRS': '1',
            'PYTHONWARNINGS': 'ignore',
            'MPLBACKEND': 'Agg'
        })
        self._km = KernelManager(
            shell_port=ports[0],
            iopub_port=ports[1],
            stdin_port=ports[2],
            hb_port=ports[3],
            control_port=ports[4],
            env=env
        )
        self._km.start_kernel()
        self._owns_kernel = True
        self._client = self._km.client()
        self._client.start_channels()

    def __enter__(self):
        """Support for with statement."""
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        """Support for with statement, auto-close resources."""
        self.close()

    def __call__(self, code: str, timeout: Optional[float] = None):
        """Execute code with optional timeout."""
        return self.execute(code, timeout)

    def execute(self, code: str, timeout: Optional[float] = None) -> str:
        """Execute code and capture output."""
        if not self._client:
            return "[ERROR] Sandbox is closed."

        effective_timeout = timeout or self._default_timeout
        self._client.execute(code)

        start_time = time.time()
        stdout_parts = []
        stderr_parts = []

        while True:
            elapsed = time.time() - start_time
            if elapsed > effective_timeout:
                self._km.interrupt_kernel()
                return f'[ERROR] Execution timed out after {effective_timeout} seconds.'

            try:
                msg = self._client.get_iopub_msg(timeout=0.2)
                msg_type = msg['header']['msg_type']
                content = msg['content']

                if msg_type == 'stream':
                    target = stdout_parts if content['name'] == 'stdout' else stderr_parts
                    target.append(content['text'])

                elif msg_type == 'error':
                    stderr_parts.append(self._format_error(content['traceback']))

                elif msg_type in {'execute_result', 'display_data'}:
                    if 'data' in content and 'text/plain' in content['data']:
                        text = content['data']['text/plain']
                        stdout_parts.append(text if text.endswith('\n') else f'{text}\n')

                elif msg_type == 'status' and content.get('execution_state') == 'idle':
                    break

            except queue.Empty:
                continue
            except Exception as e:
                stderr_parts.append(f"[INTERNAL ERROR] {str(e)}")
                break

        return self._build_output(stdout_parts, stderr_parts)

    def _build_output(self, stdout_list: List[str], stderr_list: List[str]) -> str:
        """Format and merge output results."""
        stdout = "".join(stdout_list).strip()
        stderr = "".join(stderr_list).strip()

        if stderr:
            return f"{stdout}\n[STDERR]\n{stderr}".strip()
        return stdout if stdout else "[WARN] No output. Use print() or expression at the end."

    def _format_error(self, traceback_list: List[str]) -> str:
        """Format error messages, removing干扰 paths."""
        cleaned = [re.sub(r'\x1b\[[0-9;]*m', '', line) for line in traceback_list]
        filtered = [line for line in cleaned if 'IPython' not in line]
        return "\n".join(filtered)

    def close(self, suppress=contextlib.suppress):
        """Close the sandbox and release resources."""
        if self._client:
            with suppress(Exception):
                self._client.stop_channels()
                self._client = None

        if self._owns_kernel and self._km is not None:
            with suppress(Exception):
                self._km.shutdown_kernel(now=True)
            with suppress(Exception):
                self._km.cleanup_resources()
            self._km = None
            self._owns_kernel = False

    def reset(self):
        """Reset sandbox namespace."""
        reset_code = (
            '%reset -f\n'
            'import math, numpy as np, sympy, itertools\n'
            'from mpmath import mp\n'
            'mp.dps = 64\n'
        )
        self.execute(reset_code)

    def __del__(self):
        """Fallback cleanup."""
        self.close()

In [12]:
# =============================================================================
# Section 6: KaggleSolver Class
# =============================================================================
# Main solver class that handles model loading, inference, and tool execution

class KaggleSolver:
    """
    Kaggle AIMO3 Math Solver.
    Supports both local models (transformers) and online API models (OpenAI-compatible).
    """

    def __init__(
            self,
            model_path: str,
            data: Dict[str, Any] = None,
            base_url: str = None,
            api_key: str = None,
            max_turns: int = 128,
            max_tries: int = 1,
            enable_thinking: bool = False,
            max_context_length: int = 4096,
            train_before_inference: bool = True,
            use_peft_train: bool = True,
            peft_weight_save_path: str = "peft_adapter",
            quantize_base_model: bool = True,
            peft_config: Dict[str, Any] = None,
            sft_config: Dict[str, Any] = None,
            quantization_config: Dict[str, Any] = None,
            **kwargs
    ):
        self.model_path = model_path
        self.model = None
        self.tokenizer = None
        self.streamer = None
        self.dataset = data
        
        # Determine offline vs online mode
        self.offline_mode = True
        if base_url is not None:
            self.offline_mode = False
            self.client = OpenAI(base_url=base_url, api_key=api_key)
        
        self.max_turns = max_turns
        self.max_tries = max_tries
        self.enable_thinking = enable_thinking
        self.max_context_length = max_context_length
        self.train_before_inference = train_before_inference and self.offline_mode
        self.use_peft_train = use_peft_train
        self.peft_weight_save_path = peft_weight_save_path
        self.quantize_base_model = quantize_base_model
        self.peft_config = peft_config
        self.sft_config = sft_config
        self.quantization_config = quantization_config
        self.model_kwargs = kwargs
        self.system_prompt = SYSTEM_PROMPT
        self.tools = None

    def load(self):
        """Load the model and tokenizer."""
        print("Loading model...")
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_path,
            quantization_config=self.quantization_config,
            **self.model_kwargs
        )
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_path)
        # Set up streamer for real-time output
        self.streamer = TextIteratorStreamer(self.tokenizer, skip_prompt=True, skip_special_tokens=True)
        self.model.config.use_cache = True
        print(f"Successfully loaded model from {self.model_path}.")

    def bind_tools(self, tools):
        """Bind tools to the solver."""
        self.tools = {tool.name: tool for tool in tools}

    def tool_exec(self, tool_call_str):
        """Execute a tool call."""
        tool_call = json.loads(tool_call_str)["function"]
        name = tool_call["name"]
        kwargs = tool_call["arguments"]
        kwargs = json.loads(kwargs)
        if name not in self.tools:
            return f"ERROR: Tool {name} not found."
        result = self.tools[name](**kwargs)
        return str(result)

    def reply_iter(self, conv):
        """Generate a reply for the given conversation."""
        if self.offline_mode:
            # Local model inference
            prompt = self.tokenizer.apply_chat_template(
                conv,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=self.enable_thinking,
                tools=[]
            )
            inputs = self.tokenizer(prompt, return_tensors="pt").to("cuda")
            
            # Use thread for streaming
            kwargs = dict(**inputs, streamer=self.streamer, max_length=self.max_context_length)
            thread = Thread(target=self.model.generate, kwargs=kwargs)
            thread.start()
            
            content, reasoning_content, tool_calls = collect_model_stream(self.streamer)
            message = construct_completion_message(content, reasoning_content, tool_calls)
        else:
            # API-based inference
            response = self.client.chat.completions.create(
                messages=conv,
                model=self.model_path,
                max_tokens=self.max_context_length,
                extra_body={"enable_thinking": self.enable_thinking},
                tools=[tool.get_tool_schema() for tool in self.tools.values()],
                stream=True
            )
            content, reasoning_content, tool_calls = collect_api_stream(response)
            message = construct_completion_message(content, reasoning_content, tool_calls)

        return message

    def solve_problem(self, problem: str):
        """Solve a math problem using multi-turn conversation with tools."""
        conv = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": f"Problem: {problem}\n\nPreference: {PREFERENCE_PROMPT}"},
        ]
        answer = None
        
        for t in range(self.max_turns):
            message = self.reply_iter(conv)
            conv.append(message)
            
            # Handle tool calls
            if message.tool_calls is not None:
                tool_call = message.tool_calls[0]
                result = self.tool_exec(tool_call.model_dump_json())
                tool_msg = {"role": "tool", "content": result}
                print(f"Tool Exec: \n{result}")
                conv.append(tool_msg)
            # Check for final answer
            elif "\\boxed" in (message.content or ""):
                answer = extract_answer(message.content)
                break
            else:
                # Continue conversation
                usr_msg = {"role": "user", "content": "continue"}
                conv.append(usr_msg)

        return answer, conv

    def multiple_check_answer(self, problem: str):
        """Solve problem multiple times and return the most common answer."""
        answers = []
        for _ in range(self.max_tries):
            answer, _ = self.solve_problem(problem)
            answers.append(answer)
        counter = Counter(answers)
        if 0 in counter:
            counter.pop(0)
        if len(counter) == 0:
            return 0
        return counter.most_common(1)[0][0]

    def predict(self, problem: str):
        """Main prediction function for Kaggle inference server."""
        if self.model is None and self.offline_mode:
            self.load()
        if self.train_before_inference:
            # Training would go here if enabled
            self.train_before_inference = False
        answer = self.multiple_check_answer(problem)
        return answer

In [20]:
# =============================================================================
# Section 7: Configuration Loading
# =============================================================================
# Load configuration from YAML file
cfg_str = \
"""
data: &data
  path: json
  data_files: ../sample_3500.jsonl
  # /kaggle/input/datasets/heymipolar/openr1-sample-3500/sample_3500.jsonl
  split: train

solver:
  model_path: qwen3.5-plus
  data: *data
  # 加载模型
  base_url: https://dashscope.aliyuncs.com/compatible-mode/v1
  api_key: sk-33a563b623104cef9c730e2dcd12a395
  max_context_length: 32768
  enable_thinking: true
  max_turns: 128
"""


# Load config from local_config.yaml
# config_path = "config/online_config.yaml"
# with open(config_path, 'r') as f:
#     config = yaml.safe_load(f)
config = yaml.safe_load(cfg_str)

# Extract solver parameters
solver_params = config.get('solver', {})
print("Configuration loaded:")
print(f"  Model path: {solver_params.get('model_path')}")
print(f"  Base URL: {solver_params.get('base_url')}")
print(f"  Max turns: {solver_params.get('max_turns')}")
print(f"  Enable thinking: {solver_params.get('enable_thinking')}")

Configuration loaded:
  Model path: qwen3.5-plus
  Base URL: https://dashscope.aliyuncs.com/compatible-mode/v1
  Max turns: 128
  Enable thinking: True


In [15]:
# =============================================================================
# Section 8: Initialize KaggleSolver
# =============================================================================
# Create the solver instance with configuration

# Create solver instance (without loading dataset)
model = KaggleSolver(**solver_params)

# Bind the sandbox tool
timeout = 60  # seconds
sandbox = AIMO3Sandbox(timeout=timeout)
model.bind_tools([sandbox])

print("KaggleSolver initialized successfully!")
print(f"  Offline mode: {model.offline_mode}")
print(f"  Tools bound: {list(model.tools.keys())}")

KaggleSolver initialized successfully!
  Offline mode: False
  Tools bound: ['python_sandbox']


In [ ]:
# =============================================================================
# Section 9: Prediction Service Setup
# =============================================================================
# Set up the Kaggle inference server for competition submission

# Define the prediction function required by Kaggle
def predict(id_: pl.Series, problem: pl.Series) -> pl.DataFrame | pd.DataFrame:
    """Make a prediction for a single problem."""
    # Unpack values
    id_ = id_.item(0)
    problem_text: str = problem.item(0)
    
    # Make a prediction
    # The model is loaded on the first call
    prediction = model.predict(problem_text)
    
    return pl.DataFrame({'id': id_, 'answer': prediction})


# Create the inference server
inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(
    predict
)

# Start the server
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # For competition rerun, you MUST call this within 15 minutes
    # This ensures a "fast fail" in case of bugs
    inference_server.serve()
else:
    # For local testing
    inference_server.run_local_gateway(
        ('/kaggle/input/ai-mathematical-olympiad-progress-prize-3/test.csv',)
    )

In [16]:
# =============================================================================
# Section 10: Test (Optional - for local testing)
# =============================================================================
# Uncomment to test locally without the inference server
test_problem = """
If it takes 10 seconds for a clock to strike 5, how long does it take to strike 10?
Use python_sandbox to verify your answer first and then give your final answer.
"""
# 
# # Load model if not loaded
# if model.model is None:
#     model.load()
# 
# Get prediction
answer = model.predict(test_problem)
print(f"Answer: {answer}")

Answer: 22
